# EDGAR Financial Document Collector

This notebook uses the **SEC EDGAR API** to automatically download financial filings for a set of companies.

### What is EDGAR?
EDGAR (Electronic Data Gathering, Analysis, and Retrieval) is the SEC's public database of every regulatory filing made by US-listed companies. It's completely free and requires no API key.

### What we're collecting
| Form | What it is | Frequency |
|---|---|---|
| **10-K** | Annual report — full year financials + risk factors + MD&A | Yearly |
| **10-Q** | Quarterly report — abbreviated version of the 10-K | 3x per year |
| **8-K / Exhibit 99.1** | Earnings press release filed within 4 days of results | Per quarter |

### Output
All files are saved to `datasets/client/` with structured filenames like `NVDA_10-K_2024-01-28.htm`.

## 1. Setup

In [ ]:
import requests
import time
import json
from pathlib import Path
from datetime import datetime

# ── Output directory ──────────────────────────────────────────────────────────
OUTPUT_DIR = Path("datasets/client")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── SEC requires a User-Agent identifying you ─────────────────────────────────
# Without this header, EDGAR will block your requests.
# Format: "Name email@address.com"
HEADERS = {
    "User-Agent": "Finance RAG Project traceropt@gmail.com",
    "Accept-Encoding": "gzip, deflate",
}

# ── Rate limit: SEC allows max 10 requests/second ─────────────────────────────
REQUEST_DELAY = 0.15  # seconds between requests (~6-7 req/sec, safely under limit)

def get(url: str) -> requests.Response:
    """Wrapper that enforces the rate limit on every request."""
    time.sleep(REQUEST_DELAY)
    resp = requests.get(url, headers=HEADERS, timeout=30)
    resp.raise_for_status()
    return resp

print("Setup complete.")

## 2. Define Companies and Filing Types

Edit the `COMPANIES` list to add or remove companies. You only need the ticker symbol — CIK lookup is handled automatically in the next step.

`MAX_PER_TYPE` controls how many of each filing type to download per company (most recent first).

In [ ]:
# Tickers to collect. Add any US-listed company ticker here.
COMPANIES = ["NVDA", "AMD", "AMZN", "META", "GOOGL", "TSLA", "MSFT", "AAPL"]

# Filing types to download
FILING_TYPES = ["10-K", "10-Q", "8-K"]

# How many of each filing type to pull per company
MAX_PER_TYPE = {
    "10-K": 2,   # last 2 annual reports
    "10-Q": 4,   # last 4 quarters
    "8-K":  4,   # last 4 earnings releases
}

## 3. CIK Lookup

EDGAR identifies companies by a **CIK** (Central Index Key) number, not by ticker symbol.

EDGAR publishes a full ticker → CIK mapping at a single JSON endpoint. We download it once and build a lookup table.

In [ ]:
def build_ticker_to_cik() -> dict[str, str]:
    """
    Downloads the SEC's master ticker→CIK map and returns it as a dict.
    CIKs are zero-padded to 10 digits, which is the format the submissions
    API expects.
    """
    url = "https://www.sec.gov/files/company_tickers.json"
    data = get(url).json()
    # The JSON is a dict of dicts: {"0": {"cik_str": 320193, "ticker": "AAPL", ...}}
    return {
        entry["ticker"].upper(): str(entry["cik_str"]).zfill(10)
        for entry in data.values()
    }

ticker_to_cik = build_ticker_to_cik()

# Verify our companies are found
for ticker in COMPANIES:
    cik = ticker_to_cik.get(ticker, "NOT FOUND")
    print(f"  {ticker:6s} → CIK {cik}")

## 4. Fetch Filing List

For each company, EDGAR exposes a submissions endpoint that returns every filing ever made — form type, date, accession number, and the primary document filename.

```
https://data.sec.gov/submissions/CIK{cik}.json
```

We filter this down to the form types we care about.

In [ ]:
def get_filings(cik: str, form_type: str, max_results: int) -> list[dict]:
    """
    Returns a list of recent filings for a given CIK and form type.
    Each item is a dict with: accession_number, date, primary_document.
    """
    url = f"https://data.sec.gov/submissions/CIK{cik}.json"
    data = get(url).json()

    recent = data["filings"]["recent"]
    forms   = recent["form"]
    dates   = recent["filingDate"]
    accnums = recent["accessionNumber"]
    docs    = recent["primaryDocument"]

    results = []
    for form, date, accnum, doc in zip(forms, dates, accnums, docs):
        if form == form_type:
            results.append({
                "accession_number": accnum,
                "date": date,
                "primary_document": doc,
            })
        if len(results) >= max_results:
            break

    return results


# Quick test — show NVDA's most recent 10-K filings
nvda_cik = ticker_to_cik["NVDA"]
print("NVDA recent 10-Ks:")
for f in get_filings(nvda_cik, "10-K", 3):
    print(f"  {f['date']}  {f['accession_number']}  →  {f['primary_document']}")

## 5. Resolve the Document URL

Large-cap companies (NVDA, AAPL, MSFT, etc.) file their 10-Ks and 10-Qs as **HTML with inline XBRL** — they don't attach PDFs to EDGAR. This is standard practice for modern filings.

Our strategy:
1. Check the filing's document index for a `.pdf` attachment
2. If none found, fall back to the primary document (`.htm` / `.html`) — same content, different format

The filing index lives at:
```
https://www.sec.gov/Archives/edgar/data/{cik}/{accession_no_dashes}/{accession_no_dashes}-index.json
```

In [ ]:
def resolve_document(cik: str, accession_number: str, primary_document: str) -> tuple[str, str]:
    """
    Returns (url, file_extension) for the best available document in a filing.

    Preference order:
      1. PDF — attached to the filing when available
      2. HTM/HTML — the primary document (always present)

    Both formats contain the same regulatory content.
    """
    acc_nodash = accession_number.replace("-", "")
    base_url = f"https://www.sec.gov/Archives/edgar/data/{int(cik)}/{acc_nodash}/"

    # Check filing index for a PDF attachment
    index_url = f"{base_url}{acc_nodash}-index.json"
    try:
        index = get(index_url).json()
        for item in index.get("directory", {}).get("item", []):
            name = item.get("name", "")
            if name.lower().endswith(".pdf"):
                return base_url + name, ".pdf"
    except Exception:
        pass  # index fetch failed — fall through to primary document

    # Fall back to the primary document (HTML)
    ext = Path(primary_document).suffix or ".htm"
    return base_url + primary_document, ext


# Test on the first NVDA 10-K
test_filing = get_filings(nvda_cik, "10-K", 1)[0]
url, ext = resolve_document(nvda_cik, test_filing["accession_number"], test_filing["primary_document"])
print(f"Format : {ext}")
print(f"URL    : {url}")

## 6. Download a Filing

With a URL in hand, we download and save it. The filename encodes the ticker, form type, and date so the file is self-describing.

Re-running this cell skips files that already exist on disk, so it's safe to run multiple times.

In [ ]:
def download_filing(ticker: str, form_type: str, date: str, url: str, ext: str) -> Path | None:
    """
    Downloads a filing and saves it to OUTPUT_DIR.
    Filename format: TICKER_FORM-TYPE_DATE.ext  (e.g. NVDA_10-K_2024-01-28.htm)
    Returns the saved path, or None on error.
    """
    filename = f"{ticker}_{form_type.replace('/', '-')}_{date}{ext}"
    dest = OUTPUT_DIR / filename

    if dest.exists():
        print(f"  [skip] {filename}")
        return dest

    try:
        resp = get(url)
        dest.write_bytes(resp.content)
        size_kb = dest.stat().st_size / 1024
        print(f"  [OK]   {filename}  ({size_kb:.0f} KB)")
        return dest
    except Exception as e:
        print(f"  [ERR]  {filename}: {e}")
        return None

## 7. Run the Collector

Now we wire everything together. For each company and filing type:
1. Look up the CIK
2. Get the list of recent filings
3. Resolve the best available document URL (PDF preferred, HTML fallback)
4. Download and save

**Note on 8-K filings:** Not every 8-K is an earnings release — companies file 8-Ks for executive changes, acquisitions, and other material events too. The earnings press release is typically `Exhibit 99.1` of the 8-K. For simplicity we download all recent 8-Ks here.

In [ ]:
downloaded = []
errors     = []

for ticker in COMPANIES:
    cik = ticker_to_cik.get(ticker)
    if not cik:
        print(f"\n{ticker}: CIK not found, skipping")
        continue

    print(f"\n{'─'*50}")
    print(f"{ticker} (CIK {cik})")
    print(f"{'─'*50}")

    for form_type in FILING_TYPES:
        max_n = MAX_PER_TYPE[form_type]
        filings = get_filings(cik, form_type, max_n)
        print(f"  {form_type}: {len(filings)} filings found")

        for filing in filings:
            url, ext = resolve_document(cik, filing["accession_number"], filing["primary_document"])
            path = download_filing(ticker, form_type, filing["date"], url, ext)
            if path:
                downloaded.append(path)
            else:
                errors.append(f"{ticker} {form_type} {filing['date']}")

print(f"\n{'='*50}")
print(f"Done. {len(downloaded)} files saved, {len(errors)} errors.")

## 8. Summary

Review what was collected. The summary table covers both the files just downloaded and any pre-existing files in `datasets/client/`.

In [ ]:
import pandas as pd

rows = []
for f in sorted(OUTPUT_DIR.iterdir()):
    if not f.is_file():
        continue
    parts = f.stem.split("_")  # TICKER_FORM_DATE
    rows.append({
        "file": f.name,
        "format": f.suffix,
        "ticker": parts[0] if len(parts) >= 3 else "?",
        "form":   parts[1] if len(parts) >= 3 else "?",
        "date":   parts[2] if len(parts) >= 3 else "?",
        "size_kb": round(f.stat().st_size / 1024, 1),
    })

df = pd.DataFrame(rows)
print(f"Total files : {len(df)}")
print(f"Total size  : {df['size_kb'].sum() / 1024:.1f} MB")
print(f"Formats     : {df['format'].value_counts().to_dict()}")
df

In [ ]:
if errors:
    print("Failed downloads (may need manual retrieval):")
    for e in errors:
        print(f"  {e}")
else:
    print("No errors.")